# 01 Download

Downloads the challenge files from the bucket `prc-2026-datasets` (access keys from the organisers, read by `pyopensky`
from its configuration file) into `data/01_raw`, then merges the twelve monthly training files into one departure file
and one arrival file, both sorted by scheduled time.

Every run fetches all files again, so a re-uploaded file is always picked up. Later steps never write to
`data/01_raw`; they save their results in their own folders.

In [1]:
import os
from pathlib import Path

# The challenge files carry R arrow metadata (extension type "arrow.r.vctrs"); its columns load as their plain type.
os.environ["POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR"] = "load_as_storage"
import polars as pl
from pyopensky.s3 import S3Client

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
RAW = ROOT / "data/01_raw"
RAW.mkdir(parents=True, exist_ok=True)
BUCKET = "prc-2026-datasets"

In [2]:
s3 = S3Client().s3client
for item in s3.list_objects(BUCKET, recursive=True):
    if item.object_name.endswith(".parquet"):
        print("downloading", item.object_name, item.last_modified)
        s3.fget_object(BUCKET, item.object_name, str(RAW / Path(item.object_name).name))
for path in sorted(RAW.glob("*.parquet")):
    print(f"{path.name:45s} {path.stat().st_size / 1024**2:8.1f} MiB")

downloading final_ranking.parquet 2026-10-08 11:04:30.040000+00:00


downloading final_submitting.parquet 2026-10-08 11:04:39.095000+00:00


downloading ranking.parquet 2026-09-04 08:30:40.896000+00:00


downloading submitting.parquet 2026-09-04 08:27:23.547000+00:00
downloading training_2025-01-01_2025-02-01.parquet 2026-08-13 08:06:57.137000+00:00


downloading training_2025-02-01_2025-03-01.parquet 2026-08-13 08:06:59.019000+00:00


...

downloading training_2025-09-01_2025-10-01.parquet 2026-08-13 08:06:41.506000+00:00


downloading training_2025-10-01_2025-11-01.parquet 2026-08-13 08:06:58.672000+00:00


downloading training_2025-11-01_2025-12-01.parquet 2026-08-13 08:06:25.369000+00:00


downloading training_2025-12-01_2026-01-01.parquet 2026-08-13 08:06:57.822000+00:00


final_ranking.parquet                             81.9 MiB
final_submitting.parquet                           2.9 MiB
ranking.parquet                                   41.5 MiB
submitting.parquet                                 1.6 MiB
train_ARR_merged_sorted.parquet                   91.4 MiB
train_DEP_merged_sorted.parquet                   87.3 MiB
training_2025-01-01_2025-02-01.parquet            20.2 MiB
training_2025-02-01_2025-03-01.parquet            18.9 MiB
training_2025-03-01_2025-04-01.parquet            21.6 MiB
training_2025-04-01_2025-05-01.parquet            22.7 MiB
training_2025-05-01_2025-06-01.parquet            24.0 MiB
training_2025-06-01_2025-07-01.parquet            23.8 MiB
training_2025-07-01_2025-08-01.parquet            24.8 MiB
training_2025-08-01_2025-09-01.parquet            24.8 MiB
training_2025-09-01_2025-10-01.parquet            23.9 MiB
training_2025-10-01_2025-11-01.parquet            24.2 MiB
training_2025-11-01_2025-12-01.parquet            21.1 M

## Merge the training months

In [3]:
monthly = sorted(RAW.glob("training_*.parquet"))
training = pl.concat([pl.scan_parquet(p) for p in monthly], how="vertical_relaxed")
for phase, name in [("DEP", "train_DEP_merged_sorted.parquet"), ("ARR", "train_ARR_merged_sorted.parquet")]:
    training.filter(pl.col("PHASE_mvt").str.to_uppercase() == phase).sort("SCHED_TIME_UTC_mvt", maintain_order=True) \
        .sink_parquet(RAW / name, compression="zstd")
    print(name, pl.scan_parquet(RAW / name).select(pl.len()).collect().item(), "rows")

train_DEP_merged_sorted.parquet 2085047 rows


train_ARR_merged_sorted.parquet 2082750 rows
